# 07 · 학습 결과 시각화 (HuggingFace 체크포인트 기반)

> **공부 기록 노트북 7번.** 학습은 RunPod(GPU)에서 하고, **결과는 HuggingFace에
> 저장**해 두었습니다. 이 노트북은 그 저장본을 받아 와서 **Colab(무료 T4면 충분,
> CPU로도 됨)** 에서 예측을 그려 봅니다. 시각화는 추론 몇 장이라 GPU가 거의
> 안 듭니다.

## 흐름

```
HuggingFace (duckbin/surgical-sam2-temporal)   ← 학습 결과 zip 저장됨
   │  hf download + unzip
   ▼
outputs/<model>/best.ckpt 복원
   │  + CholecSeg8k 데이터(입력·정답 프레임용)
   ▼
[input | GT | 모델 예측] 나란히 그림 + per-class Dice 막대
```

## 이 노트북의 특징

- **있는 체크포인트만** 자동으로 찾아 그립니다. 지금은 `sam2_temporal` 하나뿐이라
  3열(`input | GT | temporal`)만 나오고, 나중에 U-Net·SAM2까지 학습해 올리면
  열이 자동으로 늘어납니다.
- HF repo가 **private** 이면 토큰이 필요합니다(아래 설명).

⚠️ 가장 오래 걸리는 건 **CholecSeg8k 다운로드(~3GB, 20–40분)** 입니다 — 정답
마스크를 그려야 해서 필요합니다.

## 0. 환경 준비 — 코드 받기

GitHub `main` 에서 코드를 받습니다 (idempotent — 여러 번 실행해도 안전).

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
!pip -q install huggingface_hub segmentation-models-pytorch transformers peft \
    hydra-core omegaconf \
    albumentations pytorch-lightning datasets matplotlib >/dev/null 2>&1
# Colab ships an old torchao that breaks current peft's LoRA dispatch;
# we don't use torchao quantization, so remove it to skip that path.
!pip -q uninstall -y torchao >/dev/null 2>&1

import torch
print("준비 완료 ·", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (시각화는 CPU로도 OK)")

## 1. HuggingFace 에서 학습 결과 받기

RunPod 에서 `hf upload` 로 올려둔 zip 을 내려받아 풉니다. 풀면
`outputs/<model>/best.ckpt` 가 복원되어, 아래 셀들이 그대로 읽습니다.

- **`HF_REPO`** 를 본인 것으로 바꾸세요 (기본: `duckbin/surgical-sam2-temporal`).
- repo 가 **private** 이면 먼저 토큰으로 로그인해야 합니다. 아래 `HF_TOKEN` 에
  Write/Read 토큰(https://huggingface.co/settings/tokens)을 넣거나, 공개 repo면
  비워 두세요.

In [ ]:
from huggingface_hub import hf_hub_download, login
import zipfile, os

HF_REPO  = "duckbin/surgical-sam2-temporal"   # ← 본인 repo
HF_FILE  = "sam2_temporal_results.zip"          # 업로드할 때 쓴 파일명
HF_TOKEN = ""                                    # private 이면 토큰, 공개면 "" 로

if HF_TOKEN:
    login(token=HF_TOKEN)

zip_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE,
                           repo_type="model",
                           token=HF_TOKEN or None)
print("받음:", zip_path)

# /content/surgical-ai 아래로 풀기 → outputs/, results/ 복원
with zipfile.ZipFile(zip_path) as z:
    z.extractall(".")
print("풀린 항목:")
for root, _, files in os.walk("outputs"):
    for f in files:
        print("  ", os.path.join(root, f))
# train.log / RUN_INFO.txt 는 workspace/ 아래로 풀릴 수 있음 (무해)


## 2. CholecSeg8k 데이터 (입력·정답 프레임)

예측을 정답(GT)·입력과 나란히 그리려면 원본 프레임이 필요합니다. HuggingFace
캐시로 받습니다. **이 셀이 가장 오래 걸립니다 (~3GB, 20–40분).** 한 번 받으면
같은 세션에선 다시 안 받습니다.

In [ ]:
!bash scripts/download_cholecseg8k.sh

## 3. 어떤 모델이 받아졌는지 자동 감지 + 로드

`outputs/<name>/best.ckpt` 가 있는 모델만 골라 로드합니다. 지금은 보통
`sam2_temporal` 하나뿐이고, 나중에 다른 모델을 학습해 같은 repo 에 올리면
여기서 자동으로 함께 잡힙니다.

In [ ]:
import torch
from pathlib import Path
from omegaconf import OmegaConf

from src.train.train_segmentation import build_model
from src.eval.benchmark_runner import load_segmentation_checkpoint

CANDIDATES = [
    ("U-Net",              "unet_baseline"),
    ("SAM2+LoRA",          "sam2_lora"),
    ("SAM2+LoRA+temporal", "sam2_temporal"),
]
device = "cuda" if torch.cuda.is_available() else "cpu"

loaded = []   # (label, name, cfg, model)
for label, name in CANDIDATES:
    ckpt = Path(f"outputs/{name}/best.ckpt")
    if not ckpt.is_file():
        print(f"[skip] {label}: 체크포인트 없음 ({ckpt})")
        continue
    cfg = OmegaConf.load(f"configs/model/{name}.yaml")
    model = load_segmentation_checkpoint(build_model(cfg, pretrained=False), str(ckpt))
    loaded.append((label, name, cfg, model.to(device).eval()))
    print(f"[ok]   {label} 로드됨")

assert loaded, "받아진 체크포인트가 없습니다 — 1번(HF 다운로드)을 확인하세요."
print(f"\n로드된 모델 {len(loaded)}개:", [l[0] for l in loaded])

## 4. (핵심) 나란히 비교 — `[input | GT | <각 모델 예측>]`

같은 val 프레임을 로드된 모든 모델에 먹여 한 줄로 비교합니다. 색은
`src/utils/viz.py` 의 6-class 팔레트: 빨강=liver, 파랑=gallbladder,
**노랑=cystic_duct**(관심 대상), 초록=cystic_artery, 회색=tool.

`SAMPLE_INDICES` 로 다른 프레임을 골라볼 수 있습니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.data.cholecseg8k import CholecSeg8kDataset, CholecSeg8kWindowDataset
from src.data.transforms import build_eval_transforms
from src.utils.viz import overlay_mask

IMG = 256
SAMPLE_INDICES = [0, 50, 120]

eval_tf = build_eval_transforms(IMG)
frame_ds = CholecSeg8kDataset(split="val", image_size=IMG, transform=eval_tf)

# temporal 모델이 있으면 같은 타깃 프레임의 윈도우도 준비
has_temporal = any(c.get("window") for _, _, c, _ in loaded)
window_ds = None
if has_temporal:
    win = int(next(c.get("window", 3) for _, _, c, _ in loaded if c.get("window")))
    window_ds = CholecSeg8kWindowDataset(split="val", window=win,
                                         image_size=IMG, transform=eval_tf)

def _denorm(img_t):
    mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])
    arr = img_t.permute(1, 2, 0).numpy() * std + mean
    return (np.clip(arr, 0, 1) * 255).astype(np.uint8)

@torch.no_grad()
def _pred_frame(model, sample):
    return model(sample["image"].unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()

ncols = 2 + len(loaded)
fig, ax = plt.subplots(len(SAMPLE_INDICES), ncols,
                       figsize=(4 * ncols, 4 * len(SAMPLE_INDICES)))
if len(SAMPLE_INDICES) == 1:
    ax = ax[np.newaxis, :]
titles = ["input", "ground truth"] + [l[0] for l in loaded]

for r, idx in enumerate(SAMPLE_INDICES):
    sample = frame_ds[idx]
    rgb = _denorm(sample["image"]); gt = sample["mask"].numpy()
    ax[r, 0].imshow(rgb)
    ax[r, 1].imshow(overlay_mask(rgb, gt))
    for c, (label, name, cfg, model) in enumerate(loaded, start=2):
        if cfg.get("window") and window_ds is not None:
            # temporal: 타깃이 이 프레임인 윈도우를 찾아 사용
            match = next((window_ds[w] for w in range(len(window_ds))
                          if window_ds._windows[w][-1] == idx), None)
            pred = _pred_frame(model, match) if match else gt * 0
        else:
            pred = _pred_frame(model, sample)
        ax[r, c].imshow(overlay_mask(rgb, pred))
    if r == 0:
        for c, t in enumerate(titles):
            ax[r, c].set_title(t)
    for c in range(ncols):
        ax[r, c].axis("off")
plt.tight_layout(); plt.show()

## 5. per-class Dice 막대 비교

zip 에 함께 받아진 `results/benchmark_table.md` 를 파싱해 클래스별 Dice 를
막대로 그립니다. **cystic_duct** 막대를 특히 보세요 (지금은 0일 수 있음).

In [ ]:
import re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

path = Path("results/benchmark_table.md")
if not path.is_file():
    print("results/benchmark_table.md 없음 — zip 에 포함됐는지 확인하세요.")
else:
    section = path.read_text().split("Per-class Dice")[-1]
    rows = [r.strip() for r in section.splitlines() if r.strip().startswith("|")]
    rows = [r for r in rows if not set(r.replace("|", "").strip()) <= set("-: ")]
    labels = [h.strip() for h in rows[0].strip("|").split("|")][1:]
    cls, mat = [], []
    for r in rows[1:]:
        parts = [p.strip() for p in r.strip("|").split("|")]
        cls.append(parts[0])
        mat.append([float(m.group(1)) if (m := re.match(r"([0-9.]+)", c)) else np.nan
                    for c in parts[1:]])
    mat = np.array(mat)
    x = np.arange(len(cls)); w = 0.8 / max(1, len(labels))
    plt.figure(figsize=(11, 5))
    for i, lab in enumerate(labels):
        plt.bar(x + (i - len(labels)/2 + 0.5) * w, mat[:, i], w, label=lab)
    plt.xticks(x, cls, rotation=15); plt.ylim(0, 1.0); plt.ylabel("Dice")
    plt.title("Per-class Dice (CholecSeg8k test)"); plt.legend()
    plt.grid(axis="y", ls=":", alpha=0.4); plt.tight_layout(); plt.show()

## 마무리

- **4번(나란히 비교)**: 모델 예측이 정답을 얼마나 따라가는지 *눈으로*.
- **5번(막대)**: 클래스별 강·약점을 *숫자로*. cystic_duct 가 0 이면 학습이
  희소 클래스를 못 잡은 것 — early stopping 을 늦추거나 손실 가중치를 키워
  재학습하면 개선됩니다.

### 다음
- U-Net·SAM2 도 학습해 같은 HF repo 에 올리면, 이 노트북이 자동으로 3·4·5열
  비교표를 채웁니다.
- 개선된 재학습 후 새 zip 을 올리고 1번 `HF_FILE` 만 바꾸면 그대로 다시 그릴
  수 있습니다.